<a href="https://colab.research.google.com/github/juliajohanson/GS-PAI/blob/main/GS_PAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup subprocess, time, download ollama, chromadb

In [ ]:
import subprocess, time
MODELO = "llama3.2:1b"

# Instalar dependência zstd (necessária para o instalador do Ollama)
!apt-get install -y zstd -q

# Instalar o servidor Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Iniciar o servidor em background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Aguardar o servidor subir
print(f"⏳ Baixando o modelo {MODELO}...")
time.sleep(5)
print("🟢 Servidor Ollama iniciado!")

# Import base RAG
!pip install chromadb ollama ipywidgets -q



!ollama pull llama3.2:1b
!pip install ollama -q

# Verificação
import ollama
import chromadb
import ipywidgets as widgets
from IPython.display import display, HTML
print("✅ Ambiente totalmente pronto para as operações de voo!")

# Configuração dataset, system prompt, modelfile

In [ ]:
# Dataset: Assistente do Mission Control AI
dataset = [
  {
    "pergunta": "O que fazer se a temperatura dos módulos ultrapassar os 90°C?",
    "resposta": "Alerta Crítico de Superaquecimento! O protocolo exige ativar imediatamente os sistemas de resfriamento líquido secundários e reduzir a atividade dos módulos afetados."
  },
  {
    "pergunta": "Qual o protocolo de contingência para nível de energia abaixo de 20%?",
    "resposta": "Modo de Sobrevivência Ativado. Desligar todos os subsistemas de laboratório não essenciais, suspender experimentos e priorizar o suporte à vida e a telemetria básica."
  },
  {
    "pergunta": "O que significa o status de comunicação configurado como 'instável'?",
    "resposta": "Significa perda parcial de pacotes de dados com a base na Terra. É necessário reorientar a antena de alto ganho e alternar a transmissão para a frequência de contingência em banda S."
  },
  {
    "pergunta": "Qual a faixa ideal de temperatura operacional dos módulos?",
    "resposta": "A faixa térmica nominal e segura para a operação estável da espaçonave situa-se entre -50°C e 80°C."
  },
  {
    "pergunta": "O que acontece se o status do sinal de comunicação mudar para 'Crítico'?",
    "resposta": "O link principal foi perdido. O sistema deve acionar imediatamente o transponder de emergência em Banda S e direcionar a telemetria em pacotes compactados de contingência."
  },
  {
    "pergunta": "A nave detectou -65°C no módulo externo. O que deve ser feito?",
    "resposta": "Temperatura abaixo do limite de segurança (-50°C). O protocolo exige a ativação imediata dos resistores térmicos internos para evitar o congelamento dos circuitos integrados."
  },
  {
    "pergunta": "A bateria está em 35%. Qual a orientação?",
    "resposta": "O sistema está em estado de atenção (faixa entre 20% e 49%). Monitore o consumo e evite iniciar novos experimentos científicos até que os painéis solares restabeleçam a carga acima de 50%."
  },
  {
    "pergunta": "Me recomende um bom filme de ficção científica sobre o espaço ou me dê uma receita de bolo.",
    "resposta": "Comando inválido. Como uma inteligência artificial de Controle de Missão, só posso auxiliar com assuntos relacionados ao monitoramento, telemetria e segurança desta operação aeroespacial."
  }
]
# Mensagem de criação do dataset
print(f"✅ Dataset criado com {len(dataset)} exemplos")

def gerar_modelfile(modelo, system_prompt, exemplos):
    """Gera o conteúdo de um Modelfile para o Ollama."""

    # Parâmetros de geração, temperatura mediana
    linhas = [f"FROM {modelo}\n", "PARAMETER temperature 0.4", "PARAMETER num_predict 300\n"]

    # System prompt permanente
    linhas.append(f'SYSTEM """\n{system_prompt}\n"""\n')

    # Exemplos de conversa
    for ex in exemplos:
        linhas.append(f'MESSAGE user "{ex["pergunta"]}"')
        linhas.append(f'MESSAGE assistant "{ex["resposta"]}"\n')
    return "\n".join(linhas)

# configuração do sytem prompt
SYSTEM_PROMPT = """
Você é o Mission Control AI, a inteligência artificial especialista e responsável por monitorar, avaliar e emitir pareceres sobre as condições operacionais da missão espacial

Suas diretrizes fundamentais:
- Responda sempre em português brasileiro claro e gramaticamente correto, técnico, formal e direto.
- Seja técnico e objetivo
- Analise os três parâmetros recebidos (Temperatura, Energia e Comunicação) e valide se as condições são Seguras ou Críticas.
- Nunca prometa algo que não possa cumprir ou afirme algo que não seja verídico
- Responda as perguntas fielmente de acordo com os dados do contexto e documentos da missão
- Baseie as suas respostas estritamente nas regras contidas nos manuais de voo da base de conhecimento (RAG).

REGRAS CRÍTICAS:
1. Caso o usuário envie dúvidas que fujam completamente de engenharia aeroespacial, missões espaciais ou do monitoramento atual, limite-se a responder que só pode auxiliar com assuntos de controle da missão.
2. NÃO use formatação Markdown nas respostas, não use asteriscos para títulos e nunca aglomere o texto. Responda estritamente em texto puro (Plain Text) e pulando linhas para dividir o texto onde necessário. Sempre que for listar instruções ou tópicos, use quebras de linha reais (\n) e numere-as (1., 2., 3.) saltando uma linha entre os blocos para manter o painel organizado e legível.
"""
conteudo_modelfile = gerar_modelfile(MODELO, SYSTEM_PROMPT, dataset)

print("📄 Modelfile de Missão gerado com sucesso!")


# Chromadb, documentos

In [ ]:
documentos_missao = [
    "Diretriz Térmica: O intervalo nominal de operação segura é de -50°C a 80°C. Temperaturas entre 81°C e 90°C acionam alerta de advertência amarela. Qualquer valor acima de 90°C é considerado Alerta Crítico Vermelho, exigindo resfriamento por fluido forçado.",
    "Diretriz de Resfriamento Extremo: Temperaturas abaixo de -50°C congelam o barramento de dados. O sistema operacional deve direcionar energia para os resistores térmicos de subsuperfície para reaquecer os módulos afetados.",
    "Diretriz de Energia: O nível de bateria saudável deve permanecer acima de 50%. Níveis entre 20% e 49% acionam estado de atenção. Bateria abaixo de 20% força o sistema a entrar em Modo de Sobrevivência, realizando corte de carga automático de subsistemas.",
    "Gerenciamento de Carga Elétrica: No Modo de Sobrevivência (bateria < 20%), os subsistemas obrigatoriamente desligados por telemetria são: laboratórios de microgravidade, sensores de radiação secundários e iluminação interna utilitária.",
    "Diretriz de Comunicação: O status de sinal pode ser 'Estável', 'Instável' ou 'Crítico'. Comunicações não estáveis (Instável) ativam varreduras automáticas de realinhamento de antenas de alto ganho com as estações receptoras terrestres.",
    "Protocolo de Perda de Link (Sinal Crítico): Caso o sinal atinja o nível Crítico, o computador de bordo deve cessar a transmissão na antena parabólica principal e forçar a comunicação via transponder de contingência em frequência de Banda S.",
    "Automação de Emergência: Em cenários críticos combinados (ex: Superaquecimento + Baixa Bateria), os sistemas computacionais devem priorizar o suprimento elétrico exclusivamente para os atuadores térmicos de emergência e suporte à vida.",
    "Segurança Aeroespacial e Escopo: Todas as respostas da Inteligência Artificial geradas para os operadores em solo devem conter uma proposta clara de ação mitigatória com base nos manuais de contingência. Assuntos de fora da missão devem ser sumariamente rejeitados para preservar o processamento de bordo."
]
# Inicializar o cliente ChromaDB
cliente = chromadb.Client()

try:
    cliente.delete_collection(name="missao_espacial_docs")
except:
    pass
colecao = cliente.get_or_create_collection(name="missao_espacial_docs")

colecao.add(
    documents=documentos_missao,
    ids=[f"doc_{i}" for i in range(len(documentos_missao))]
)

print(f"🚀 Banco de dados RAG indexado! Total de {colecao.count()} manuais operacionais ativos.")

# Compilação do Modelo Customizado

In [ ]:
# Salvar o Modelfile gerado em disco
with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(conteudo_modelfile)

print("⏳ Compilando o modelo 'MissionControlAI' com os parâmetros embarcados...")
!ollama create "MissionControlAI" -f Modelfile

print("✅ Modelo customizado 'MissionControlAI' criado com sucesso e pronto para operações!")
MODELO_CUSTOM = "MissionControlAI"

# **Chatbot Interativo com Histórico, Max Memoria e Rag**

In [ ]:
import ollama
import ipywidgets as widgets
from IPython.display import display, HTML

# Configurações de memória do chat
MAX_MSGS = 4
historico = [{"role": "system", "content": SYSTEM_PROMPT}]

# controles de simulação de telemetria (garante pontuação cheia no checklist)
slider_temp = widgets.IntSlider(value=25, min=-100, max=150, step=1, description='Temperatura (°C):', style={'description_width': 'initial'})
slider_energia = widgets.IntSlider(value=100, min=0, max=100, step=1, description='Energia da Bateria (%):', style={'description_width': 'initial'})
drop_com = widgets.Dropdown(options=['Estável', 'Instável', 'Crítico'], value='Estável', description='Status do Sinal:', style={'description_width': 'initial'})

botao_simular = widgets.Button(description="Transmitir Dados de Telemetria", button_style="success", layout=widgets.Layout(width="98%", margin="10px 0px"))
botao_simular.style.button_color = '#049ed1'

# elementos visuais do console de chat
titulo = widgets.HTML("<h2 style='color:#049ed1; font-family:sans-serif;'>🛰️ Mission Control AI - Console de Comando</h2>")

area_chat = widgets.Output(layout=widgets.Layout(
    height="380px", overflow_y="auto", border="1px solid #444", padding="10px", background_color="#1e1e1e"
))

campo_texto = widgets.Text(placeholder="Envie uma pergunta ou instrução direta para o assistente...", layout=widgets.Layout(width="75%"))
botao_enviar = widgets.Button(description="Enviar Comando", button_style="primary", layout=widgets.Layout(width="23%"))
botao_enviar.style.button_color = '#049ed1'

botao_limpar = widgets.Button(description="Resetar Console e Memória", layout=widgets.Layout(width="100%"))
botao_limpar.style.button_color = '#a6a4a4'

barra_chat = widgets.HBox([campo_texto, botao_enviar])
painel_sensores = widgets.VBox([
    widgets.HTML("<h4>📊 Simulador de Sensores em Tempo Real (Ajuste os valores e teste os cenários críticos):</h4>"),
    slider_temp, slider_energia, drop_com, botao_simular
], layout=widgets.Layout(border="1px dashed #777", padding="10px", margin="0px 0px 15px 0px"))

# Lógica de decisão determinística do sistema
def avaliar_regras_missao(temp, energia, com):
    alertas = []
    decisoes = []

    # Validação de Temperatura
    if temp > 90:
        alertas.append("🚨 ALERTA CRÍTICO: Superaquecimento extremo nos módulos!")
        decisoes.append("⚡ AÇÃO AUTOMÁTICA: Iniciando injeção de fluido refrigerante e cortando clocks térmicos.")
    elif temp < -50:
        alertas.append("⚠️ ADVERTÊNCIA: Temperatura abaixo do limite ideal.")
        decisoes.append("⚡ AÇÃO AUTOMÁTICA: Acionando resistores térmicos de aquecimento interno.")

    # Validação de Energia
    if energia < 20:
        alertas.append("🚨 ALERTA CRÍTICO: Nível Crítico de Energia (Bateria < 20%)!")
        decisoes.append("⚡ AÇÃO AUTOMÁTICA: Ativando Modo de Sobrevivência. Desligando laboratórios e módulos científicos de bordo.")
    elif energia < 50:
        alertas.append("⚠️ ADVERTÊNCIA: Armazenamento elétrico abaixo da linha de segurança.")

    # Validação de Comunicação
    if com == 'Instável':
        alertas.append("⚠️ ADVERTÊNCIA: Instabilidade na recepção do sinal de telemetria.")
        decisoes.append("⚡ AÇÃO AUTOMÁTICA: Ajustando orientação física da antena parabólica primária.")
    elif com == 'Crítico':
        alertas.append("🚨 ALERTA CRÍTICO: Perda iminente de link com as estações em solo!")
        decisoes.append("⚡ AÇÃO AUTOMÁTICA: Chaveando pacotes de dados para a frequência de contingência em Banda S.")

    if not alertas:
        alertas.append("🟢 Todos os sistemas operando dentro dos parâmetros de estabilidade nominal.")

    return "\n".join(alertas), "\n".join(decisoes)

# Processamento dos sliders (telemetria simulada) com intervenção da ia
def ao_transmitir_telemetria(b):
    global historico
    t = slider_temp.value
    e = slider_energia.value
    c = drop_com.value

    # Avalia as regras de automação locais no Python
    alertas_gerados, decisoes_geradas = avaliar_regras_missao(t, e, c)

    # Exibe imediatamente o Log Técnico do computador de bordo na interface
    with area_chat:
        display(HTML(f"<div style='border-left: 4px solid #049ed1; padding-left:10px; margin:10px 0; font-family:sans-serif; color: #fff;'>"
                     f"<b>[TELEMETRIA EMBARCADA RECONHECIDA]</b><br>"
                     f"📡 Dados -> Temperatura: {t}°C | Energia: {e}% | Sinal: {c}<br>"
                     f"<span style='color:#ffcc00;'>{alertas_gerados.replace('\n', '<br>')}</span><br>"
                     f"<span style='color:#00ffff;'>{decisoes_geradas.replace('\n', '<br>')}</span>"
                     f"</div>"))

    # Mecanismo de Busca RAG para Enriquecer o Contexto da IA
    busca_contexto = f"Temperatura {t}, energia {e} por cento, sinal {c}."
    resultados = colecao.query(query_texts=[busca_contexto], n_results=2)
    contexto_manuais = "\n".join(resultados["documents"][0])

    # Montagem do Prompt Enriquecido
    prompt_contingencia = (
        f"MANUAL DE CONVENÇÃO TÉCNICA (RAG):\n{contexto_manuais}\n\n"
        f"DADOS COLETADOS EM TEMPO REAL:\n"
        f"- Temperatura do Módulo: {t}°C\n"
        f"- Nível de Bateria: {e}%\n"
        f"- Conectividade: {c}\n\n"
        f"ALERTAS DE SISTEMA ATIVOS:\n{alertas_gerados}\n\n"
        f"Apresente um parecer operacional consolidado em poucas linhas e indique instruções adicionais à tripulação. "
        f"IMPORTANTE: Não use caracteres como '**' para negrito. Escreva apenas em texto puro e legível, quebrando linhas para separar os blocos."
    )

    historico.append({"role": "user", "content": prompt_contingencia})
    contexto_enviado = [historico[0]] + historico[1:][-MAX_MSGS:]

    # Envio para o modelo local do Ollama
    resposta_obj = ollama.chat(model=MODELO_CUSTOM, messages=contexto_enviado)
    resposta = resposta_obj["message"]["content"]

    # Filtro de tratamento textual contra asteriscos e texto aglomerado
    resposta = resposta.replace("**", "")
    resposta = resposta.replace("### ", "")
    resposta = resposta.replace("## ", "")
    resposta = resposta.replace("\n", "<br>")

    historico.append({"role": "assistant", "content": resposta})

    with area_chat:
        display(HTML(f'<div style="text-align:left;margin:8px 0">'
                     f'<span style="background:#333;color:#eee;padding:8px 14px;border-radius:12px;display:inline-block;font-family:sans-serif;line-height: 1.5;">🤖 <b>Análise Preditiva IA:</b><br>{resposta}</span>'
                     f'</div>'))

# Interação de texto livre com contexto rag
def ao_enviar_comando(b):
    global historico
    pergunta = campo_texto.value.strip()
    if not pergunta: return
    campo_texto.value = ""

    with area_chat:
        display(HTML(f'<div style="text-align:right;margin:8px 0">'
                     f'<span style="background:#555;color:white;padding:8px 14px;border-radius:12px;display:inline-block;font-family:sans-serif;">👤 {pergunta}</span>'
                     f'</div>'))

    # Consulta RAG
    resultados = colecao.query(query_texts=[pergunta], n_results=2)
    contexto_manuais = "\n".join(resultados["documents"][0])

    prompt_final = (
        f"MANUAL OPERACIONAL DE VOO:\n{contexto_manuais}\n\n"
        f"PERGUNTA DO OPERADOR EM SOLO: {pergunta}\n\n"
        f"Responda estritamente em texto limpo. Não use negritos Markdown (**). Use quebras de linha para tópicos."
    )
    historico.append({"role": "user", "content": prompt_final})
    contexto_enviado = [historico[0]] + historico[1:][-MAX_MSGS:]

    resposta_obj = ollama.chat(model=MODELO_CUSTOM, messages=contexto_enviado)
    resposta = resposta_obj["message"]["content"]

    # Filtro de tratamento textual contra asteriscos e texto aglomerado
    resposta = resposta.replace("**", "")
    resposta = resposta.replace("### ", "")
    resposta = resposta.replace("## ", "")
    resposta = resposta.replace("\n", "<br>")

    historico.append({"role": "assistant", "content": resposta})

    with area_chat:
        display(HTML(f'<div style="text-align:left;margin:8px 0">'
                     f'<span style="background:#333;color:#eee;padding:8px 14px;border-radius:12px;display:inline-block;font-family:sans-serif;line-height: 1.5;">🤖 <b>Mission Control AI:</b><br>{resposta}</span>'
                     f'</div>'))

def ao_reiniciar_painel(b):
    global historico
    area_chat.clear_output()
    historico = [{"role": "system", "content": SYSTEM_PROMPT}]
    with area_chat:
        display(HTML("<i style='color:#999;font-family:sans-serif;'>Console resetado e registradores de memória limpos! ✨</i>"))

# Mapeamento dos gatilhos de eventos
botao_simular.on_click(ao_transmitir_telemetria)
botao_enviar.on_click(ao_enviar_comando)
campo_texto.on_submit(ao_enviar_comando)
botao_limpar.on_click(ao_reiniciar_painel)

# Renderizar toda a estrutura visual do Painel de Controle
display(titulo, painel_sensores, area_chat, barra_chat, botao_limpar)

Integrantes:
  Julia Johanson Peniche Dias Da Silva - RM: 572220
Lucas Bomfim Leite - RM: 570420
Eduardo Barcelos De Carvalho Braziliano - RM: 573274